In [11]:
import torch 


# input of layer 2
bs = 16 # batch size

Leye = torch.rand(bs,128, 8, 8)
print(Leye)
Reye = torch.rand(bs,128, 8, 8)
print(Reye)
FaceData = torch.rand(bs,128, 8, 8)  # currently matched with eyes......
print(FaceData)

tensor([[[[0.4252, 0.7302, 0.1403,  ..., 0.0889, 0.4432, 0.4035],
          [0.6137, 0.7911, 0.7287,  ..., 0.9910, 0.8427, 0.1499],
          [0.4313, 0.7731, 0.7468,  ..., 0.8234, 0.4803, 0.3618],
          ...,
          [0.8179, 0.9129, 0.5005,  ..., 0.9573, 0.7073, 0.9350],
          [0.4349, 0.7207, 0.6254,  ..., 0.4195, 0.0039, 0.8796],
          [0.7859, 0.0800, 0.4563,  ..., 0.6110, 0.3865, 0.6078]],

         [[0.5446, 0.4935, 0.3615,  ..., 0.7323, 0.0034, 0.1910],
          [0.5031, 0.2311, 0.9162,  ..., 0.0789, 0.0659, 0.6243],
          [0.3355, 0.4308, 0.8100,  ..., 0.4602, 0.1761, 0.5743],
          ...,
          [0.7482, 0.6834, 0.3144,  ..., 0.0122, 0.2467, 0.1407],
          [0.3948, 0.6891, 0.8166,  ..., 0.4444, 0.0203, 0.6305],
          [0.9715, 0.5442, 0.8804,  ..., 0.8849, 0.1292, 0.4849]],

         [[0.0031, 0.1870, 0.6912,  ..., 0.3033, 0.4854, 0.6295],
          [0.3349, 0.3500, 0.5004,  ..., 0.0238, 0.6045, 0.0280],
          [0.3260, 0.1573, 0.1764,  ..., 0

In [ ]:
class TinyModel(torch.nn.Module):
    def __init__(self):
        super(TinyModel, self).__init__()
        # layer 1: feature extraction (to be implemented)
        
        
        # layer 2: feature fusion: concate + group  normalization
        # self.concate = torch.cat((x, x, x), 0) // not needed here, only in forward
        # total_ch = Leye[1]+Reye[1]+FaceData[1]  # total channels of the input // this is not working 
        total_ch = 384
        self.gn = torch.nn.GroupNorm(3, total_ch)     # Separate 6 channels into 3 groups, how to define a appropriate number of groups & channels?
        
        # layer 3:self-attention
        # self.norm = torch.nn.LayerNorm([total_ch, 8, 8])  # here is the problem!!!  it normalizes the whole input, not the channel
        self.norm = torch.nn.LayerNorm(total_ch)  # only the channel should be normalized
        self.mha = torch.nn.MultiheadAttention(total_ch, num_heads=1, batch_first=True)  # heads = 1, now it's single head
        self.scale = torch.nn.Parameter(torch.zeros(1))

    def use_attention(self, x): 
        # ####################################################################
        # !!!! now we dont have position encoding!!!! but question is, do we need a compelete transformer? or just self-attention is enough?
        # also there is no MLP and the 2nd Norm, necessary?
        ########################################################################

        # Reshape input for multi-head attention
        bs, c, h, w = x.shape
        x_att = x.reshape(bs, c, h * w).transpose(1, 2)  # BSxHWxC    # For MHA, the input shoulb be (batch_size, sequence_length, embedding_dim)
        
        # Apply Layer Normalization
        x_att = self.norm(x_att)
        Q = K = V = x_att
        # Apply Multi-head Attention
        att_out, att_map  = self.mha(Q, K, V)  # Q,K,V  Q=K=V, self-attention
        return att_out.transpose(1, 2).reshape(bs, c, h, w), att_map # transpose first, then Reshape output to original shape (bs, c, h, w)


    def forward(self, left_eye, right_eye, face):

        # layer2
        concate = torch.cat((left_eye, right_eye, face), 1)  # dim = 0 or 1?  only channel dim changes?
        out = self.gn(concate)
        # out = out.unsqueeze(bs)  
        att_out, _ = self.use_attention(out)
        out = out + self.scale * att_out
        return out 
        

model = TinyModel()
print(model)

output = model(Leye, Reye, FaceData)  # currently the face Data is not matched with eyes, should be matched in the future, how to match? linear padding or other methods? would it affect the performance if change the size of facedata
print("output:",output.shape)
# testing only forward pass, not backward pass


TinyModel(
  (gn): GroupNorm(3, 384, eps=1e-05, affine=True)
  (norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
  (mha): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=384, out_features=384, bias=True)
  )
)
output: torch.Size([16, 384, 8, 8])
